# RetailMind AI — Complete Day 1 to Day 5 Notebook
## Customer Churn Prediction: Classical ML, Tuning, Feature Engineering, Ensembles and ANN

This single notebook consolidates the work for Days 1–5 using the Telco Customer Churn dataset.

**Day 1:** Load data, inspect it, preprocess it, train a Logistic Regression baseline and evaluate it.  
**Day 2:** Compare Decision Tree, Random Forest and SVM using the same train/test split.  
**Day 3:** Tune classical models with GridSearchCV and compare training/validation performance.  
**Day 4:** Engineer domain features, apply feature selection, and compare Random Forest with XGBoost.  
**Day 5:** Run a Keras MNIST warm-up, train a churn ANN with early stopping, and compare it with the best classical model.

Run all cells from top to bottom. The notebook expects the Kaggle Telco CSV in `data/WA_Fn-UseC_-Telco-Customer-Churn.csv`.

## 0. Install dependencies

In [ ]:
# Run this cell if any required library is missing:
# %pip install pandas numpy scikit-learn joblib matplotlib xgboost tensorflow


## 1. Imports and project paths

In [ ]:
from pathlib import Path
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, roc_auc_score,
    classification_report, confusion_matrix
)

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
DATA_PATH = PROJECT_ROOT / "data" / "WA_Fn-UseC_-Telco-Customer-Churn.csv"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"
MODELS_DIR.mkdir(exist_ok=True)
REPORTS_DIR.mkdir(exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Dataset path:", DATA_PATH)
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. Put the Kaggle CSV inside the project's data/ folder."
    )


## 2. Load and inspect the dataset — Day 1

In [ ]:
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
display(df.head())
display(df.info())
display(df.describe(include="all").T)
display(df.isna().sum().sort_values(ascending=False).head(15))
print("Target distribution:")
display(df["Churn"].value_counts())
display(df["Churn"].value_counts(normalize=True).rename("proportion"))


## 3. Clean data and create train/test split — Day 1

In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# customerID is an identifier, so it is excluded from model features.
X = df.drop(columns=["Churn", "customerID"]).copy()
y = df["Churn"].map({"No": 0, "Yes": 1}).astype("int32")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train churn rate:", round(y_train.mean(), 4))
print("Test churn rate:", round(y_test.mean(), 4))


## 4. Build reusable preprocessing pipeline — Day 1

In [ ]:
def make_preprocessor(X_reference):
    numeric_columns = X_reference.select_dtypes(include=["number"]).columns.tolist()
    categorical_columns = X_reference.select_dtypes(
        include=["object", "category", "bool"]
    ).columns.tolist()

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    return ColumnTransformer([
        ("num", numeric_pipeline, numeric_columns),
        ("cat", categorical_pipeline, categorical_columns)
    ])

preprocessor = make_preprocessor(X_train)


## 5. Day 1 — Logistic Regression baseline

In [ ]:
baseline_model = Pipeline([
    ("preprocessor", make_preprocessor(X_train)),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced"))
])
baseline_model.fit(X_train, y_train)

def evaluate_classifier(model, X_eval, y_eval, model_name):
    predictions = model.predict(X_eval)
    probabilities = model.predict_proba(X_eval)[:, 1]
    return {
        "Model": model_name,
        "Accuracy": accuracy_score(y_eval, predictions),
        "Precision": precision_score(y_eval, predictions, zero_division=0),
        "Recall": recall_score(y_eval, predictions, zero_division=0),
        "ROC-AUC": roc_auc_score(y_eval, probabilities)
    }

baseline_metrics = evaluate_classifier(
    baseline_model, X_test, y_test, "Logistic Regression Baseline"
)
display(pd.DataFrame([baseline_metrics]))
print(classification_report(y_test, baseline_model.predict(X_test), zero_division=0))

joblib.dump(baseline_model, MODELS_DIR / "churn_pipeline.pkl")
pd.DataFrame([baseline_metrics]).to_csv(REPORTS_DIR / "day1_baseline_metrics.csv", index=False)
print("Saved Day 1 baseline model and metrics.")


## 6. Day 2 — Compare classical models

In [ ]:
day2_models = {
    "Decision Tree": DecisionTreeClassifier(
        max_depth=6, min_samples_split=10,
        class_weight="balanced", random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, class_weight="balanced",
        random_state=RANDOM_STATE, n_jobs=-1
    ),
    "SVM": SVC(
        kernel="rbf", probability=True,
        class_weight="balanced", random_state=RANDOM_STATE
    )
}

day2_pipelines = {}
day2_results = [baseline_metrics]

for name, estimator in day2_models.items():
    pipe = Pipeline([
        ("preprocessor", make_preprocessor(X_train)),
        ("classifier", estimator)
    ])
    pipe.fit(X_train, y_train)
    day2_pipelines[name] = pipe
    day2_results.append(evaluate_classifier(pipe, X_test, y_test, name))

day2_comparison = pd.DataFrame(day2_results).sort_values(
    ["Recall", "ROC-AUC"], ascending=False
).reset_index(drop=True)
display(day2_comparison)
day2_comparison.to_csv(REPORTS_DIR / "day2_comparison.csv", index=False)

day2_champion_name = day2_comparison.loc[0, "Model"]
day2_champion = (
    baseline_model if day2_champion_name == "Logistic Regression Baseline"
    else day2_pipelines[day2_champion_name]
)
joblib.dump(day2_champion, MODELS_DIR / "day2_champion.pkl")
print("Day 2 champion:", day2_champion_name)


## 7. Day 3 — Hyperparameter tuning with GridSearchCV

In [ ]:
# Tune a Decision Tree and Random Forest. GridSearchCV uses training data only.
day3_candidates = {
    "Decision Tree": (
        Pipeline([
            ("preprocessor", make_preprocessor(X_train)),
            ("classifier", DecisionTreeClassifier(
                class_weight="balanced", random_state=RANDOM_STATE
            ))
        ]),
        {
            "classifier__max_depth": [3, 5, 8, None],
            "classifier__min_samples_split": [2, 10, 20]
        }
    ),
    "Random Forest": (
        Pipeline([
            ("preprocessor", make_preprocessor(X_train)),
            ("classifier", RandomForestClassifier(
                class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1
            ))
        ]),
        {
            "classifier__n_estimators": [100, 200],
            "classifier__max_depth": [None, 8, 12],
            "classifier__min_samples_split": [2, 5]
        }
    )
}

day3_searches = {}
day3_results = []

for name, (pipe, grid) in day3_candidates.items():
    print("Tuning:", name)
    search = GridSearchCV(
        pipe, grid, scoring="recall", cv=5, n_jobs=-1, refit=True
    )
    search.fit(X_train, y_train)
    day3_searches[name] = search

    best_model = search.best_estimator_
    metrics = evaluate_classifier(best_model, X_test, y_test, name + " Tuned")
    metrics["Best CV Recall"] = search.best_score_
    metrics["Best Parameters"] = str(search.best_params_)
    day3_results.append(metrics)

day3_comparison = pd.DataFrame(day3_results).sort_values(
    ["Best CV Recall", "ROC-AUC"], ascending=False
).reset_index(drop=True)
display(day3_comparison)
day3_comparison.to_csv(REPORTS_DIR / "day3_tuning_results.csv", index=False)

day3_champion_name = day3_comparison.loc[0, "Model"].replace(" Tuned", "")
day3_champion = day3_searches[day3_champion_name].best_estimator_
joblib.dump(day3_champion, MODELS_DIR / "churn_v3.joblib")
print("Day 3 tuned champion:", day3_champion_name)


### Day 3 — Train versus validation scores

In [ ]:
# Compare the best CV recall with held-out test recall as a practical diagnostic.
display(day3_comparison[["Model", "Best CV Recall", "Recall", "ROC-AUC"]])

print("Interpretation guide:")
print("- High training/CV performance with weaker validation/test performance can indicate overfitting.")
print("- Low scores on both training/CV and test can indicate underfitting.")
print("- A large gap is a warning sign; use cross-validation and learning curves for deeper diagnosis.")


## 8. Day 4 — Domain feature engineering

In [ ]:
class RetailFeatureEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X["TotalCharges"] = pd.to_numeric(X["TotalCharges"], errors="coerce")

        X["TenureGroup"] = pd.cut(
            X["tenure"], bins=[-1, 12, 24, 48, np.inf],
            labels=["New", "Medium", "Long", "Loyal"]
        ).astype("object")

        safe_tenure = X["tenure"].replace(0, np.nan)
        X["AverageMonthlySpend"] = X["TotalCharges"] / safe_tenure
        X["ChargeRatio"] = X["MonthlyCharges"] / (X["TotalCharges"] + 1)

        X["IsMonthToMonth"] = (X["Contract"] == "Month-to-month").astype(int)
        X["IsOneYearContract"] = (X["Contract"] == "One year").astype(int)
        X["IsTwoYearContract"] = (X["Contract"] == "Two year").astype(int)

        service_cols = [
            "PhoneService", "MultipleLines", "OnlineSecurity", "OnlineBackup",
            "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"
        ]
        X["ServiceCount"] = sum((X[col] == "Yes").astype(int) for col in service_cols)
        X["HasInternet"] = (X["InternetService"] != "No").astype(int)
        X["HasPhone"] = (X["PhoneService"] == "Yes").astype(int)
        X["PaperlessBillingFlag"] = (X["PaperlessBilling"] == "Yes").astype(int)
        return X

# Inspect engineered columns. This is only a preview; pipelines below fit transforms on each fold.
feature_engineer = RetailFeatureEngineer()
X_train_engineered = feature_engineer.fit_transform(X_train)
display(X_train_engineered.head())
print("Feature count before:", X_train.shape[1])
print("Feature count after:", X_train_engineered.shape[1])


## 9. Day 4 — Train Random Forest and XGBoost with feature selection

In [ ]:
# XGBoost is imported only for this Day 4 section.
from xgboost import XGBClassifier

negative_count = int((y_train == 0).sum())
positive_count = int((y_train == 1).sum())
scale_pos_weight = negative_count / positive_count

def make_day4_pipeline(classifier):
    # Feature engineering and preprocessing are fitted within the pipeline,
    # avoiding leakage across cross-validation folds.
    sample_engineered = RetailFeatureEngineer().fit_transform(X_train)
    numeric_cols = sample_engineered.select_dtypes(include=["number"]).columns.tolist()
    categorical_cols = sample_engineered.select_dtypes(
        include=["object", "category", "bool"]
    ).columns.tolist()

    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])
    prep = ColumnTransformer([
        ("num", num_pipe, numeric_cols),
        ("cat", cat_pipe, categorical_cols)
    ])

    return Pipeline([
        ("feature_engineering", RetailFeatureEngineer()),
        ("preprocessor", prep),
        ("feature_selection", SelectKBest(score_func=f_classif, k=40)),
        ("classifier", classifier)
    ])

day4_models = {
    "Random Forest Day 4": RandomForestClassifier(
        n_estimators=250, max_depth=12, min_samples_split=5,
        class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1
    ),
    "XGBoost": XGBClassifier(
        n_estimators=250, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight, eval_metric="logloss",
        random_state=RANDOM_STATE, n_jobs=-1
    )
}

day4_pipelines = {}
day4_results = []

for name, estimator in day4_models.items():
    print("Training:", name)
    pipe = make_day4_pipeline(estimator)

    # Determine a safe k based on transformed training data.
    X_eng = RetailFeatureEngineer().fit_transform(X_train)
    # Build a temporary preprocessor with the same columns to calculate encoded width.
    num_cols = X_eng.select_dtypes(include=["number"]).columns.tolist()
    cat_cols = X_eng.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
    temp_preprocessor = ColumnTransformer([
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                          ("scaler", StandardScaler())]), num_cols),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                          ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat_cols)
    ])
    encoded_width = temp_preprocessor.fit_transform(X_eng).shape[1]
    pipe.set_params(feature_selection__k=min(40, encoded_width))

    cv = cross_val_score(pipe, X_train, y_train, cv=5, scoring="recall", n_jobs=1)
    pipe.fit(X_train, y_train)
    day4_pipelines[name] = pipe

    metrics = evaluate_classifier(pipe, X_test, y_test, name)
    metrics["CV Recall"] = cv.mean()
    metrics["CV Recall Std"] = cv.std()
    day4_results.append(metrics)
    print(f"CV Recall: {cv.mean():.4f}")

day4_comparison = pd.DataFrame(day4_results).sort_values(
    ["CV Recall", "ROC-AUC"], ascending=False
).reset_index(drop=True)
display(day4_comparison)
day4_comparison.to_csv(REPORTS_DIR / "day4_model_comparison.csv", index=False)


## 10. Day 4 — Compare the tuned Day 3 champion and select a classical champion

In [ ]:
# Score the Day 3 model on the same untouched test set.
day3_metrics = evaluate_classifier(
    day3_champion, X_test, y_test, "Day 3 Tuned Champion"
)

classical_candidates = {
    "Day 3 Tuned Champion": day3_champion,
    **day4_pipelines
}

classical_comparison_rows = [day3_metrics]
for name, model in day4_pipelines.items():
    classical_comparison_rows.append(
        evaluate_classifier(model, X_test, y_test, name)
    )

classical_comparison = pd.DataFrame(classical_comparison_rows).sort_values(
    ["Recall", "ROC-AUC"], ascending=False
).reset_index(drop=True)
display(classical_comparison)
classical_comparison.to_csv(REPORTS_DIR / "classical_champion_comparison.csv", index=False)

# Select by held-out recall for reporting/benchmarking. Treat this test-set choice
# as a final benchmark, not as a new hyperparameter-tuning loop.
classical_champion_name = classical_comparison.loc[0, "Model"]
classical_champion = classical_candidates[classical_champion_name]
joblib.dump(classical_champion, MODELS_DIR / "phase1_classical_champion.joblib")
print("Saved Phase 1 classical champion:", classical_champion_name)


## 11. Day 5 — Keras MNIST warm-up demo

In [ ]:
import tensorflow as tf
tf.keras.utils.set_random_seed(RANDOM_STATE)

(x_train_mnist, y_train_mnist), (x_test_mnist, y_test_mnist) = tf.keras.datasets.mnist.load_data()
x_train_mnist = x_train_mnist.astype("float32") / 255.0
x_test_mnist = x_test_mnist.astype("float32") / 255.0

mnist_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(28, 28)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(10, activation="softmax")
])
mnist_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
mnist_model.fit(
    x_train_mnist, y_train_mnist,
    epochs=3, batch_size=128, validation_split=0.1, verbose=1
)
mnist_loss, mnist_accuracy = mnist_model.evaluate(x_test_mnist, y_test_mnist, verbose=0)
print(f"MNIST test accuracy: {mnist_accuracy:.4f}")


## 12. Day 5 — Prepare normalized churn inputs for the ANN

In [ ]:
# Fit preprocessing on training rows only; transform test rows with the same fitted object.
ann_preprocessor = make_preprocessor(X_train)
X_train_ann = ann_preprocessor.fit_transform(X_train)
X_test_ann = ann_preprocessor.transform(X_test)

if hasattr(X_train_ann, "toarray"):
    X_train_ann = X_train_ann.toarray()
    X_test_ann = X_test_ann.toarray()

X_train_ann = np.asarray(X_train_ann, dtype="float32")
X_test_ann = np.asarray(X_test_ann, dtype="float32")

print("ANN training matrix:", X_train_ann.shape)
print("ANN test matrix:", X_test_ann.shape)


## 13. Day 5 — Build and train the feed-forward ANN

In [ ]:
ann_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train_ann.shape[1],)),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.30),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dropout(0.20),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

ann_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(name="auc")
    ]
)
ann_model.summary()

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True
)

ann_history = ann_model.fit(
    X_train_ann, y_train,
    validation_split=0.20,
    epochs=50,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)
print("Epochs completed:", len(ann_history.history["loss"]))


## 14. Day 5 — Evaluate ANN and compare with classical champion

In [ ]:
ann_prob = ann_model.predict(X_test_ann, verbose=0).ravel()
ann_pred = (ann_prob >= 0.5).astype("int32")

ann_metrics = {
    "Model": "ANN",
    "Accuracy": accuracy_score(y_test, ann_pred),
    "Precision": precision_score(y_test, ann_pred, zero_division=0),
    "Recall": recall_score(y_test, ann_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, ann_prob)
}

classical_metrics = evaluate_classifier(
    classical_champion, X_test, y_test, classical_champion_name
)

day5_comparison = pd.DataFrame([classical_metrics, ann_metrics])
display(day5_comparison)
day5_comparison.to_csv(REPORTS_DIR / "day5_ann_comparison.csv", index=False)

print("ANN classification report:")
print(classification_report(y_test, ann_pred, zero_division=0))


## 15. Day 5 — Plot learning curves and save artifacts

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(ann_history.history["loss"], label="Training loss")
plt.plot(ann_history.history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Binary cross-entropy loss")
plt.title("Day 5 ANN: Training vs Validation Loss")
plt.legend()
plt.tight_layout()
plt.savefig(REPORTS_DIR / "day5_ann_loss.png", dpi=150)
plt.show()

ann_model.save(MODELS_DIR / "churn_ann_v5.keras")
joblib.dump(ann_preprocessor, MODELS_DIR / "churn_ann_preprocessor_v5.joblib")

print("Saved ANN:", MODELS_DIR / "churn_ann_v5.keras")
print("Saved ANN preprocessor:", MODELS_DIR / "churn_ann_preprocessor_v5.joblib")
print("Saved comparison:", REPORTS_DIR / "day5_ann_comparison.csv")


## 16. README results — fill in actual scores

Copy the actual values from the result tables into your README. Do not invent or manually estimate scores.

Suggested summary:

### Day 1 — Baseline
Logistic Regression baseline with median/mode imputation, StandardScaler and OneHotEncoder.

### Day 2 — Classical model comparison
Compared Decision Tree, Random Forest and SVM on a stratified hold-out split. Recall is important because missed churners can represent lost retention opportunities.

### Day 3 — Hyperparameter tuning
Used GridSearchCV with 5-fold cross-validation and recall scoring to tune Decision Tree and Random Forest.

### Day 4 — Feature engineering and ensembles
Created tenure, charge, contract and service features. Applied SelectKBest and compared Random Forest with XGBoost.

### Day 5 — ANN
Built a feed-forward Keras ANN with normalized inputs, Dense layers, dropout, sigmoid output and EarlyStopping. Benchmarked it against the Phase 1 classical champion.

**Honest model-selection note:** Keep the classical model if the ANN does not materially improve recall/ROC-AUC or if its extra complexity is not justified. Record the measured results and trade-offs.

## 17. Final generated files

After all cells run successfully, the notebook creates:

**Models**
- `models/churn_pipeline.pkl`
- `models/day2_champion.pkl`
- `models/churn_v3.joblib`
- `models/phase1_classical_champion.joblib`
- `models/churn_ann_v5.keras`
- `models/churn_ann_preprocessor_v5.joblib`

**Reports**
- `reports/day1_baseline_metrics.csv`
- `reports/day2_comparison.csv`
- `reports/day3_tuning_results.csv`
- `reports/day4_model_comparison.csv`
- `reports/classical_champion_comparison.csv`
- `reports/day5_ann_comparison.csv`
- `reports/day5_ann_loss.png`

## 18. Push the complete project to GitHub

Run these commands from the RetailMind project root after running all notebook cells and updating README:

```bash
git status
git add notebooks/ models/ reports/ README.md
git commit -m "Add complete Day 1 to Day 5 churn ML and ANN work"
git push origin main
```

Do not commit virtual environments, credentials, or datasets unless you have permission to redistribute them.